# 02 · Data Cleaning & Feature Preparation
Turns the raw file into an analysis-ready dataset. Every decision is logged to `outputs/tables/02_cleaning_log.csv` so the process is auditable.

In [11]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("..")
FIG, TAB = ROOT / "outputs/figures", ROOT / "outputs/tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)

NAVY, TEAL, ORANGE, GREY = "#1F3A5F", "#2A9D8F", "#E76F51", "#9AA5B1"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 160, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold",
    "axes.titlesize": 12, "axes.labelsize": 10, "font.size": 10})
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 140)

In [12]:
raw = pd.read_csv(ROOT / "data/bank_marketing.csv", sep=";")
log = []
def note(step, detail, rows_after): log.append({"step": step, "detail": detail, "rows_after": rows_after})
df = raw.copy(); note("Load raw file", "bank-additional-full.csv", len(df))

## 1. Standardise names and target

In [13]:
df.columns = df.columns.str.replace(".", "_", regex=False)
df["subscribed"] = (df.y == "yes").astype(int)
df = df.drop(columns="y")
note("Rename columns", "dots -> underscores; y -> subscribed (1 = yes)", len(df))
df.columns.tolist()

['age',
 'job',
 'marital',
 'education',
 'default',
 'housing',
 'loan',
 'contact',
 'month',
 'day_of_week',
 'duration',
 'campaign',
 'pdays',
 'previous',
 'poutcome',
 'emp_var_rate',
 'cons_price_idx',
 'cons_conf_idx',
 'euribor3m',
 'nr_employed',
 'subscribed']

## 2. Remove duplicates

In [14]:
n0 = len(df); df = df.drop_duplicates().reset_index(drop=True)
note("Drop exact duplicates", f"{n0 - len(df)} rows removed", len(df))
print(f"Removed {n0 - len(df)} duplicate rows -> {len(df):,} remain")

Removed 12 duplicate rows -> 41,176 remain


## 3. Handle `unknown` and the `pdays` sentinel
* `unknown` is **kept as a category**: dropping would lose 8,597 `default` rows (21%), and unknown-status customers behave differently (see EDA).
* `pdays = 999` → `prev_contacted` flag (0/1) and `pdays_clean` (NaN when never contacted).

In [15]:
df["prev_contacted"] = (df.pdays != 999).astype(int)
df["pdays_clean"] = df.pdays.where(df.pdays != 999)
note("pdays sentinel", "999 -> prev_contacted flag + pdays_clean (NaN)", len(df))
df[["pdays", "prev_contacted", "pdays_clean"]].describe().round(2)

,pdays,prev_contacted,pdays_clean
count,41176.00,41176.00,1515.00
mean,962.46,0.04,6.01
std,186.94,0.19,3.82
min,0.00,0.00,0.00
25%,999.00,0.00,3.00
50%,999.00,0.00,6.00
75%,999.00,0.00,7.00
max,999.00,1.00,27.00


## 4. Outliers

In [16]:
q = df[["age", "duration", "campaign", "previous"]].quantile([.5, .9, .95, .99, 1]).round(1)
q

,age,duration,campaign,previous
0.50,38.0,180.0,2.0,0.0
0.90,55.0,551.0,5.0,1.0
0.95,58.0,753.0,7.0,1.0
0.99,71.0,1271.2,14.0,2.0
1.00,98.0,4918.0,56.0,7.0


`campaign` has a maximum of 56 calls to one person and `duration` reaches 4,918 s. Rather than delete real records, we **cap** `campaign` at 10 contacts (≈99th percentile territory) as a modelling feature and keep the raw column for reference. `duration` is retained only for descriptive analysis.

In [17]:
df["campaign_capped"] = df.campaign.clip(upper=10)
share = (df.campaign > 10).mean()
note("Cap campaign", f"campaign_capped = min(campaign, 10); affects {share:.2%} of rows", len(df))
print(f"{share:.2%} of rows have campaign > 10")

2.11% of rows have campaign > 10


## 5. Feature engineering

In [18]:
df["age_band"] = pd.cut(df.age, [0, 24, 34, 44, 54, 64, 120],
                        labels=["<25", "25-34", "35-44", "45-54", "55-64", "65+"]).astype(str)
job_map = {"admin.": "Office & professional", "management": "Office & professional", "technician": "Office & professional",
           "blue-collar": "Blue-collar & services", "services": "Blue-collar & services", "housemaid": "Blue-collar & services",
           "entrepreneur": "Self-employed", "self-employed": "Self-employed",
           "retired": "Retired", "student": "Student", "unemployed": "Unemployed", "unknown": "Unknown"}
df["job_group"] = df.job.map(job_map)
df["has_loan_any"] = ((df.housing == "yes") | (df.loan == "yes")).astype(int)
df["month_num"] = df.month.map({m: i for i, m in enumerate(["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"], 1)})
df["duration_band"] = pd.cut(df.duration, [-1, 60, 180, 300, 600, 10000],
                             labels=["<1 min", "1-3 min", "3-5 min", "5-10 min", "10+ min"]).astype(str)
note("Engineered features", "age_band, job_group, has_loan_any, month_num, duration_band", len(df))
df[["age", "age_band", "job", "job_group", "has_loan_any"]].sample(5, random_state=1)

,age,age_band,job,job_group,has_loan_any
39105,26,25-34,technician,Office & professional,1
17906,25,25-34,admin.,Office & professional,0
29427,52,45-54,retired,Retired,1
13841,42,35-44,blue-collar,Blue-collar & services,0
35280,36,35-44,management,Office & professional,1


## 6. Validate and save

In [19]:
assert df.isna().sum().drop("pdays_clean").sum() == 0
assert set(df.subscribed.unique()) == {0, 1}
print(f"Final shape: {df.shape}   |   conversion rate: {df.subscribed.mean():.2%}")
df.to_csv(ROOT / "data/bank_marketing_clean.csv", index=False)
cl = pd.DataFrame(log); cl.to_csv(TAB / "02_cleaning_log.csv", index=False); cl

Final shape: (41176, 29)   |   conversion rate: 11.27%


,step,detail,rows_after
0,Load raw file,bank-additional-full.csv,41188
1,Rename columns,dots -> underscores; y -> subscribed (1 = yes),41188
2,Drop exact duplicates,12 rows removed,41176
3,pdays sentinel,999 -> prev_contacted flag + pdays_clean (NaN),41176
4,Cap campaign,"campaign_capped = min(campaign, 10); affects 2...",41176
5,Engineered features,"age_band, job_group, has_loan_any, month_num, ...",41176


In [20]:
before_after = pd.DataFrame({"raw": [raw.shape[0], raw.shape[1], raw.duplicated().sum()],
                             "clean": [df.shape[0], df.shape[1], df.drop(columns=[]).duplicated().sum()]},
                            index=["rows", "columns", "duplicate rows"])
before_after.to_csv(TAB / "02_before_after.csv"); before_after

,raw,clean
rows,41188,41176
columns,21,29
duplicate rows,12,0


## Result
* 41,188 → **41,176 rows** (12 duplicates removed), **29 columns** (8 engineered features).
* No information was thrown away: `unknown` kept as a level, sentinel `999` turned into a flag, outliers capped in a *new* column rather than overwritten.
* Clean file saved to `data/bank_marketing_clean.csv` and used by notebooks 03-05.